# Lab 04: Diagnostic Training: Weight Initialization, Regularization & Vanishing Gradients

Welcome to Laboratory 04! In this lab, we address fundamental stability and generalization challenges in deep learning:
1. **Vanishing & Exploding Gradients**: Empirically measure gradient norms across deep networks initialized poorly vs. with Xavier/Kaiming He initialization.
2. **Regularization Techniques**: Implement **Dropout** ($p=0.5$), **L2 Weight Decay**, and **Gradient Clipping** to prevent overfitting and stabilize training.


## 1. Technical Preliminaries & Imports


In [ ]:
# Import required scientific packages and PyTorch modules
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import numpy as np

# Set deterministic random seed
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Active Compute Device:', device)


## 2. Vanishing Gradients Demonstration: Deep Network Gradient Diagnostics

### Conceptual & Mathematical Foundation
In a deep network with $L$ layers using Sigmoid activation $\sigma(z)$:
$$\frac{\partial \mathcal{L}}{\partial \mathbf{W}_1} = \frac{\partial \mathcal{L}}{\partial \mathbf{z}_L} \cdot \prod_{l=2}^{L} \left( \mathbf{W}_l^T \text{diag}(\sigma'(\mathbf{z}_{l-1})) \right) \mathbf{x}^T$$
Since $\max(\sigma'(z)) = 0.25$, multiplying $L$ derivative terms causes the gradient magnitude to decay exponentially ($0.25^L \to 0$), freezing early layer learning!


In [ ]:
# Build a 10-layer Deep Network to demonstrate the Vanishing Gradient problem
depth = 10
layers = []
for i in range(depth):
    layers.append(nn.Linear(30, 30))
    layers.append(nn.Sigmoid()) # Sigmoid activation causes maximum derivative of 0.25

deep_sigmoid_net = nn.Sequential(*layers)

# Synthetic batch input
x_dummy = torch.randn(64, 30)
out = deep_sigmoid_net(x_dummy)
loss = out.sum()

# Compute gradients across the deep network
loss.backward()

# Extract gradient L2 norms for each linear layer
grad_norms = []
for name, param in deep_sigmoid_net.named_parameters():
    if 'weight' in name and param.grad is not None:
        grad_norms.append(param.grad.norm().item())

# Display vanishing gradient profile across network depth
print('Gradient Norms from Input Layer (0) to Output Layer (9):')
for layer_idx, g_norm in enumerate(grad_norms):
    print(f'  Layer {layer_idx+1:02d} Gradient Norm: {g_norm:.8f}')


## 3. Regularized Architecture with Dropout & Weight Initialization

### Architecture Overview: `RegularizedClassifier`
The `RegularizedClassifier` incorporates best practices for stable deep network training:
* **Custom Initialization**: Implements Kaiming (He) Normal initialization for weights followed by zero bias initialization.
* **Dropout Regularization**: Injects stochastic zeroing of activations during training with probability $p=0.3$ to prevent co-adaptation of features.
* **L2 Regularization**: Weight decay applied through the optimizer.


In [ ]:
# Define Regularized Deep Neural Network with Dropout
class RegularizedClassifier(nn.Module):
    """Multi-layer binary classifier equipped with Dropout regularization."""
    def __init__(self, in_features: int = 30, hidden: int = 64, dropout_rate: float = 0.3):
        super(RegularizedClassifier, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),         # Randomly zero 30% of activations during training
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),         # Prevents feature co-adaptation
            nn.Linear(hidden, 1)                # Output binary logit
        )
        self._init_weights()
        
    def _init_weights(self):
        """Apply Kaiming (He) Normal initialization for ReLU layers."""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu') # He initialization
                nn.init.constant_(m.bias, 0.0)                         # Zero bias initialization
                
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

# Load and standardize Breast Cancer Wisconsin dataset
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to PyTorch DataLoaders
train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32).unsqueeze(1))
test_ds = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32).unsqueeze(1))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

# Initialize regularized model, BCEWithLogitsLoss, and Adam with weight decay (L2 penalty)
model = RegularizedClassifier(in_features=30, hidden=64, dropout_rate=0.3).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4) # weight_decay = L2 lambda

print('Initialized Regularized Model:\n', model)


### Training Routine with Gradient Norm Clipping
The training routine below demonstrates **Gradient Clipping** (`nn.utils.clip_grad_norm_`) which bounds the maximum gradient norm to prevent exploding gradients.


In [ ]:
# Train model with gradient clipping and L2 regularization
num_epochs = 25
for epoch in range(num_epochs):
    model.train() # Enable Dropout active behavior
    total_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        # Forward pass
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient Clipping: Clip gradients whose L2 norm exceeds threshold 1.0
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        total_loss += loss.item() * batch_X.size(0)
        
    if (epoch + 1) % 5 == 0:
        print(f'Epoch [{epoch+1:02d}/{num_epochs}] | Train BCE Loss: {total_loss / len(train_ds):.4f}')

# Evaluate model performance on test set
model.eval() # Disable Dropout during evaluation
correct = 0
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        logits = model(batch_X)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == batch_y).sum().item()

test_acc = (correct / len(test_ds)) * 100.0
print(f'\nTest Accuracy with Kaiming Init + Dropout + L2 Decay: {test_acc:.2f}%')


## 4. Summary & Diagnostic Best Practices
1. **Weight Initialization**: Use Kaiming (He) initialization for ReLU networks and Xavier (Glorot) initialization for Sigmoid/Tanh networks to preserve activation variance across layers.
2. **Dropout**: Randomly dropping units during training prevents complex co-adaptations and acts as an ensemble of thinned networks.
3. **Weight Decay (L2 Regularization)**: Penalizes large weight values, enforcing smoother decision boundaries.
4. **Gradient Clipping**: Prevents exploding gradients during training of deep architectures and sequence models.
